In [7]:
import numpy as np


In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Layer

class GradientReversal(Layer):
    def __init__(self, lambda_=1.0, **kwargs):
        super().__init__(**kwargs)
        self.lambda_ = lambda_

    def call(self, x):
        @tf.custom_gradient
        def _reverse(x):
            def grad(dy):
                return -self.lambda_ * dy
            return x, grad
        return _reverse(x)

    def get_config(self):
        config = super().get_config()
        config.update({"lambda_": self.lambda_})
        return config


In [10]:
import tensorflow as tf
from tensorflow.keras.layers import Layer

class Attention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        # inputs: (batch, time, features)
        score = tf.nn.softmax(tf.reduce_mean(inputs, axis=-1), axis=1)
        score = tf.expand_dims(score, axis=-1)
        context = tf.reduce_sum(inputs * score, axis=1)
        return context

    def get_config(self):
        config = super().get_config()
        return config


In [11]:
shared_features = Attention()(x)


In [12]:
from tensorflow.keras import layers, models

input_shape = (128, 300, 1)
inputs = layers.Input(shape=input_shape)

# CNN feature extractor
x = layers.Conv2D(32, (3,3), activation="relu", padding="same")(inputs)
x = layers.MaxPooling2D((2,2))(x)
x = layers.BatchNormalization()(x)

x = layers.Conv2D(64, (3,3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.BatchNormalization()(x)

# reshape for LSTM
x = layers.Reshape((x.shape[1], x.shape[2]*x.shape[3]))(x)

# temporal modeling
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)

# attention
shared_features = Attention()(x)

# 🔹 Emotion classifier
emotion_output = layers.Dense(5, activation="softmax", name="emotion")(shared_features)

# 🔹 Domain classifier (with GRL)
grl = GradientReversal(lambda_=1.0)(shared_features)
domain_hidden = layers.Dense(64, activation="relu")(grl)
domain_output = layers.Dense(2, activation="softmax", name="domain")(domain_hidden)

model = models.Model(
    inputs=inputs,
    outputs=[emotion_output, domain_output]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 300,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 128, 300,  │        320 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 64, 150,   │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 150,   │        128 │ max_pooling2d_4[… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 150,   │     18,496 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 32, 75,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 75,    │        256 │ max_pooling2d_5[… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 32, 4800)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 32, 256)   │  5,047,296 │ reshape_2[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_3         │ (None, 256)       │          0 │ bidirectional_2[… │
│ (Attention)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gradient_reversal_1 │ (None, 256)       │          0 │ attention_3[0][0] │
│ (GradientReversal)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │     16,448 │ gradient_reversa… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emotion (Dense)     │ (None, 5)         │      1,285 │ attention_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ domain (Dense)      │ (None, 2)         │        130 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,084,359 (19.40 MB)

 Trainable params: 5,084,167 (19.39 MB)

 Non-trainable params: 192 (768.00 B)

In [14]:
import pandas as pd

df_train = pd.read_csv(r"D:\SER_Cross\data\processed\train.csv")
print(df_train.head())
print(df_train["dataset"].value_counts())


                                                path  emotion   speaker  \
0  D:\SER_Cross\data\processed\ravdess_audio\Acto...  neutral  actor_02   
1  D:\SER_Cross\data\processed\ravdess_audio\Acto...  neutral  actor_02   
2  D:\SER_Cross\data\processed\ravdess_audio\Acto...  neutral  actor_02   
3  D:\SER_Cross\data\processed\ravdess_audio\Acto...  neutral  actor_02   
4  D:\SER_Cross\data\processed\ravdess_audio\Acto...  neutral  actor_02   

   dataset  
0  ravdess  
1  ravdess  
2  ravdess  
3  ravdess  
4  ravdess  
dataset
crema_d    4408
ravdess     660
Name: count, dtype: int64


In [15]:
import numpy as np

domain_labels = np.array(
    [0 if d == "ravdess" else 1 for d in df_train["dataset"]]
)

print(domain_labels.shape)
print(np.unique(domain_labels, return_counts=True))


(5068,)
(array([0, 1]), array([ 660, 4408], dtype=int64))


In [18]:
import numpy as np

X_train = np.load(r"D:\SER_Cross\features\train\X.npy")
y_train = np.load(r"D:\SER_Cross\features\train\y.npy")

# add channel dimension
X_train = X_train[..., np.newaxis]

print(X_train.shape, y_train.shape)


(5068, 128, 300, 1) (5068,)


In [19]:
print(len(domain_labels), len(X_train))


5068 5068


In [21]:
import numpy as np
import pandas as pd

# load validation features
X_val = np.load(r"D:\SER_Cross\features\val\X.npy")
y_val = np.load(r"D:\SER_Cross\features\val\y.npy")

# add channel dimension
X_val = X_val[..., np.newaxis]

# load validation CSV (for domain labels)
df_val = pd.read_csv(r"D:\SER_Cross\data\processed\val.csv")

print(X_val.shape, y_val.shape)
print(df_val["dataset"].value_counts())


(1031, 128, 300, 1) (1031,)
dataset
crema_d    811
ravdess    220
Name: count, dtype: int64


In [24]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        "emotion": "sparse_categorical_crossentropy",
        "domain": "sparse_categorical_crossentropy"
    },
    loss_weights={
        "emotion": 1.0,
        "domain": 0.3
    },
    metrics={
        "emotion": "accuracy",
        "domain": "accuracy"
    }
)


In [25]:
model.fit(
    X_train,
    {
        "emotion": y_train,
        "domain": domain_labels
    },
    validation_data=(
        X_val,
        {
            "emotion": y_val,
            "domain": np.array(
                [0 if d=="ravdess" else 1 for d in df_val["dataset"]]
            )
        }
    ),
    epochs=35,
    batch_size=32
)


Epoch 1/35
159/159 ━━━━━━━━━━━━━━━━━━━━ 105s 617ms/step - domain_accuracy: 0.1308 - domain_loss: 5.1572 - emotion_accuracy: 0.3331 - emotion_loss: 1.4680 - loss: 3.0206 - val_domain_accuracy: 0.3715 - val_domain_loss: 0.8528 - val_emotion_accuracy: 0.3026 - val_emotion_loss: 1.6310 - val_loss: 1.8739
Epoch 2/35
159/159 ━━━━━━━━━━━━━━━━━━━━ 90s 568ms/step - domain_accuracy: 0.4959 - domain_loss: 0.9089 - emotion_accuracy: 0.3861 - emotion_loss: 1.4062 - loss: 1.6797 - val_domain_accuracy: 0.5674 - val_domain_loss: 0.9504 - val_emotion_accuracy: 0.3763 - val_emotion_loss: 1.4571 - val_loss: 1.7336
Epoch 3/35
159/159 ━━━━━━━━━━━━━━━━━━━━ 78s 491ms/step - domain_accuracy: 0.6888 - domain_loss: 0.8017 - emotion_accuracy: 0.4309 - emotion_loss: 1.3422 - loss: 1.5823 - val_domain_accuracy: 0.6479 - val_domain_loss: 0.9755 - val_emotion_accuracy: 0.3628 - val_emotion_loss: 1.4569 - val_loss: 1.7396
Epoch 4/35
159/159 ━━━━━━━━━━━━━━━━━━━━ 78s 490ms/step - domain_accuracy: 0.7346 - domain_loss: 

In [28]:
import numpy as np

# load test features
X_test = np.load(r"D:\SER_Cross\features\test\X.npy")
y_test = np.load(r"D:\SER_Cross\features\test\y.npy")

# add channel dimension
X_test = X_test[..., np.newaxis]

print(X_test.shape, y_test.shape)


(1128, 128, 300, 1) (1128,)


In [29]:
emotion_pred, domain_pred = model.predict(X_test)
y_pred = np.argmax(emotion_pred, axis=1)


36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 118ms/step


In [30]:
from sklearn.metrics import accuracy_score

print("DANN Test Accuracy:", accuracy_score(y_test, y_pred))


DANN Test Accuracy: 0.5168439716312057
